# SAFE-GPT Molecular Generation Tutorial

This notebook extracts fragments from SMILES and generates molecules containing those
fragments using SAFE-GPT. Run all generation cells with the `safe` kernel.

## Fragmentation Methods
- **BRICS**: Chemically meaningful fragmentation via BRICS decomposition.
- **RC_CMS**: Random-cut fragmentation (SP3-SP3 bonds, ring-ring connections, etc.).

This notebook runs inference with an existing checkpoint. Optional training uses
the repository scripts described in Section 7.


## 0. Environment Setup

Run the following commands **in a terminal at the repository root** if the environment
is not already prepared. If it already exists, skip `conda create`.

```bash
conda create -n safe python=3.12.12 -y
conda run -n safe pip install -r requirements/safe_requirements.txt
conda run -n safe pip install -e .
conda run -n safe pip install ipykernel
conda run -n safe python -m ipykernel install --user --name safe --display-name "safe"
```

Select the **safe** kernel in Jupyter before running the cells below.
The first cell sets the working directory to the repository root so model, data,
and output paths work from this notebook's location in `tutorial/`.


In [ ]:
from pathlib import Path
import os

# Resolve paths consistently when opened from tutorial/ or the repository root.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'setup.py').is_file() and (path / 'src' / 'func').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')


## 1. Download Models

Download pre-trained models from HuggingFace Hub (`sato-akinori/FFMG`).  
For private repositories, log in beforehand with `huggingface-cli login`.

The model paths in Section 2 follow this repository's layout
(`models/{repr_name}/{model_name}/{model_ver}/...`). If the downloaded archive expands
into a different layout, move the extracted directories so they match.

In [ ]:
import os
import glob
import subprocess
from huggingface_hub import snapshot_download

# Download models from HuggingFace
snapshot_download(
    repo_id='sato-akinori/FFMG',
    allow_patterns='models/*',
    local_dir='.'
)

# Extract zip files
for zip_file in glob.glob('models/**/*.zip', recursive=True):
    subprocess.run(['unzip', '-o', zip_file, '-d', os.path.dirname(zip_file)], check=True)
    os.remove(zip_file)


# Download pretrained safe-gpt model
snapshot_download(
    repo_id='datamol-io/safe-gpt',
    local_dir='models/safe/gpt/pretrained'
)

print('Model download complete.')

## 2. Configuration

Configure the fragmentation method, generation parameters, etc.

In [ ]:
# ========================================
# Configuration (modify as needed)
# ========================================
FRAG_METHOD  = 'rc_cms'      # 'brics' or 'rc_cms'
MODEL_VER    = 'finetuning'  # 'finetuning', 'from_scratch', or 'pretrained'
N_SAMPLES    = 10            # Number of molecules to generate
NUM_BEAMS    = 10            # Number of beams for beam search
MAX_LENGTH   = 200           # Maximum sequence length
RANDOM_SEED  = 42            # Seed for rc_cms fragmentation; generation reproducibility also depends on hardware and library versions

# Model path relative to the repository root.
SAFE_MODEL_PATH = (
    'models/safe/gpt/pretrained/' if MODEL_VER == 'pretrained'
    else f'models/safe/gpt/{MODEL_VER}/{FRAG_METHOD}/best_model/'
)

### Relationship to the batch generation code

The SAFE prefix includes the trailing `.`, with BOS only (no trailing EOS),
left padding, beam search, and decoding through `encode_prefixes` and
`decode_safe_smiles` from `src/func/generation_safe_func.py`. Invalid candidates retain
their rank as `INVALID_SMILES`. The model position limit is checked before generation.

The interactive defaults are 10 candidates and 10 beams per prompt, versus 50/50
in the batch shell scripts. Inference processes one prompt at a time. Match the
checkpoint, fragment string, candidate count, and beam width for comparisons;
matching a seed alone does not guarantee identical results across environments.


## 3. Import Libraries and Helper Functions

In [ ]:
import sys
import os
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import torch
from safe.trainer.model import SAFEDoubleHeadsModel
from transformers import PreTrainedTokenizerBase
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

from func.fragmentation import (
    BRICSFragmentize,
    RandomFragmentize,
    PostProcessSelectFrags,
)


def fragmentize_smiles(
    smiles: str, frag_method: str = 'brics', ratio: float = 0.6,
    big_ring_thres: int = 7, seed: int = 42,
) -> str | None:
    """Extract one processed fragment set using the dataset's fragmentation settings.

    Args:
        smiles: Input molecule as SMILES.
        frag_method: BRICS or random-cut fragmentation.
        ratio: Fraction of eligible bonds to cut for rc_cms.
        big_ring_thres: Ring-size threshold for rc_cms.
        seed: Random-cut seed.

    Returns:
        Dot-separated fragments, with multiplicities preserved, or None.
        This interactive example keeps all processed fragments; it does not
        sample subsets or apply the training-pair size filter in Smi2Sentences.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    if frag_method == 'brics':
        frags = BRICSFragmentize(mol, returnSmiles=False)
    elif frag_method == 'rc_cms':
        frags = RandomFragmentize(
            mol, returnSmiles=False, bigRingThres=big_ring_thres,
            rseed=seed, ratio=ratio, removeDummy=False,
        )
    else:
        raise ValueError(f'Unknown method: {frag_method}')
    if frags is None:
        return None
    pass_frags, _ = PostProcessSelectFrags(
        frags, smallCarbonFilter=True,
        trimRgroupOnRing=(frag_method == 'rc_cms'),
        uniquenize=False, returnAsSmi=True,
    )
    return pass_frags


def draw_mols(smiles_list, legends=None, mols_per_row=4, img_size=(300, 300)):
    """Draw molecules from a list of SMILES strings."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    mols = [m for m in mols if m is not None]
    if not mols:
        print('No valid molecules to draw.')
        return
    if legends is None:
        legends = [Chem.MolToSmiles(m) for m in mols]
    img = Draw.MolsToGridImage(
        mols[:12], molsPerRow=mols_per_row, subImgSize=img_size, legends=legends[:12]
    )
    display(img)


def generate_safe(
    fragments: str, model: "SAFEDoubleHeadsModel", tokenizer: "PreTrainedTokenizerBase",
    device: "torch.device", n_samples: int, num_beams: int, max_length: int,
) -> list[str]:
    """Generate ranked SAFE-GPT candidates, retaining invalid prediction slots.

    The fragments are encoded as a SAFE-string prefix and the model completes the
    molecule with beam search, matching src/func/generation_safe_func.py so the results
    use the same generation procedure as the evaluation pipeline.

    Args:
        fragments: Dot-separated fragment SMILES.
        model: Loaded SAFE-GPT model in evaluation mode.
        tokenizer: SAFE tokenizer with left padding configured.
        device: Device holding the model.
        n_samples: Number of candidates, including invalid ones.
        num_beams: Beam-search width.
        max_length: Maximum total token sequence length.

    Returns:
        Ranked SMILES strings, using INVALID_SMILES for failed decodes.
    """
    import safe
    import torch

    from func.generation_safe_func import decode_safe_smiles, encode_prefixes
    from func.utility import INVALID_SMILES

    if not 1 <= n_samples <= num_beams:
        raise ValueError('Require 1 <= n_samples <= num_beams.')
    if max_length > model.config.max_position_embeddings:
        raise ValueError('max_length exceeds the model position limit.')

    prefix = safe.SAFEConverter(slicer=None).encoder(
        fragments, canonical=True, randomize=False, constraints=None, allow_empty=True
    ) + '.'
    print(f'SAFE prefix: {prefix}')

    input_ids, attention_mask = encode_prefixes([prefix], tokenizer, device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            num_beams=num_beams,
            num_return_sequences=n_samples,
            max_length=max_length,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    smiles = [decode_safe_smiles(seq) or INVALID_SMILES for seq in decoded]
    valid = [s for s in smiles if s != INVALID_SMILES]
    print(f'Generated: {len(smiles)}, Valid: {len(valid)}')

    display(pd.DataFrame({'SMILES': smiles}))
    return smiles


def load_safe_model(model_path: str) -> tuple["SAFEDoubleHeadsModel", "PreTrainedTokenizerBase", "torch.device"]:
    """Load a SAFE-GPT model for inference.

    Args:
        model_path: SAFE checkpoint directory containing model and tokenizer.

    Returns:
        Model in evaluation mode, left-padded tokenizer, and device.
    """
    import torch
    from safe.tokenizer import SAFETokenizer
    from safe.trainer.model import SAFEDoubleHeadsModel

    model = SAFEDoubleHeadsModel.from_pretrained(model_path)
    tokenizer = SAFETokenizer.from_pretrained(model_path).get_pretrained()
    # Decoder-only batched generation requires left padding.
    tokenizer.padding_side = 'left'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    print(f'Using device: {device}')
    return model, tokenizer, device


print('Library import complete.')


## 4. Input SMILES and Fragmentation

Enter an arbitrary SMILES and perform fragmentation.  
Modify `input_smiles` to try different molecules.

The cut ratio (0.6), ring threshold (7), small-carbon filtering, and ring trimming
match `src/gen_frags/rffmg_frags.py`; repeated fragments are retained. This example
uses one cut pattern and all processed fragments. Dataset construction additionally
samples subsets over multiple patterns and applies a molecule/fragment size filter.
Use the same stored fragment set when comparing with a dataset result.


In [ ]:
# ========================================
# Input SMILES (modifiable)
# ========================================
input_smiles = 'CC(C)Cc1ccc(C(C)C(=O)O)cc1'  # Ibuprofen

# Validate and canonicalize SMILES
mol = Chem.MolFromSmiles(input_smiles)
assert mol is not None, 'Invalid SMILES. Please enter a valid SMILES string.'
canonical_smi = Chem.MolToSmiles(mol)
print(f'Input molecule (canonical SMILES): {canonical_smi}')

# Fragmentation
pass_frags = fragmentize_smiles(canonical_smi, frag_method=FRAG_METHOD, seed=RANDOM_SEED)

if pass_frags is None:
    print('This molecule cannot be fragmented. Try a different SMILES or method.')
else:
    print(f'\n--- Fragmentation Results ---')
    print(f'Method: {FRAG_METHOD}')
    print(f'Fragments: {pass_frags}')
    print(f'Number of fragments: {len(pass_frags.split("."))}')

    frag_smiles = pass_frags.split('.')
    draw_mols(
        [canonical_smi] + frag_smiles,
        legends=['Input molecule'] + [f'Fragment {i+1}' for i in range(len(frag_smiles))]
    )

---
## 5. Molecule Generation from Fragmented Input

Generate molecules from the fragments obtained in Section 4.

The fragments are encoded as a SAFE-string prefix and completed with beam search,
matching `src/func/generation_safe_func.py`.


In [ ]:
assert pass_frags is not None, 'Fragmentation failed. Please check Section 4.'

safe_model, safe_tokenizer, safe_device = load_safe_model(SAFE_MODEL_PATH)

print(f'Input fragments: {pass_frags}')
valid_smiles = generate_safe(pass_frags, safe_model, safe_tokenizer, safe_device, N_SAMPLES, NUM_BEAMS, MAX_LENGTH)
if valid_smiles:
    draw_mols(valid_smiles)

---
## 6. Generate Molecules from Custom Fragments

Instead of fragmenting a molecule first, you can directly specify fragment SMILES and generate molecules.

**Fragment format:**
- Use `[*]` or `*` to mark attachment points
- Separate multiple fragments with `.` (dot)
- The full fragment string is passed to the model; no scaffold-decoration or
  scaffold-morphing API is called.

**Examples:**
- Single fragment: `c1ccc([*])cc1` (benzene with one attachment point)
- Multiple fragments: `[*]c1ccccc1.[*]C(=O)O` (benzene + carboxylic acid)

In [ ]:
# ========================================
# Input fragments directly (modifiable)
# ========================================
# Separate multiple fragments with '.'
input_fragments = ['OC1=C(O)C=CC([*])=C1.[*]N[*]', 'OC1=C(O)C=CC([*])=C1', 'OC1=C(C=CC=C1)O.[*]N[*]', 'OC1=C(C=CC=C1)O']
input_fragments = [[Chem.MolToSmiles(Chem.MolFromSmiles(frag)) for frag in input_fragment.split('.')] for input_fragment in input_fragments]

custom_frags = ['.'.join(input_fragment) for input_fragment in input_fragments]
print(f'Input fragments (canonical): {custom_frags}')
print(f'Number of fragments: {len([frag for frag_list in input_fragments for frag in frag_list])}')

for custom_frag in custom_frags:
    draw_mols(custom_frag.split('.'), legends=[f'Fragment {i+1}' for i in range(len(custom_frag.split('.')))])

### Generate from Custom Fragments

The fragments are completed with the same SAFE prefix generation used in Section 5.


In [ ]:

results = list()
safe_model, safe_tokenizer, safe_device = load_safe_model(SAFE_MODEL_PATH)

for custom_frag in custom_frags:
    print(f'Input fragments: {custom_frag}')
    smiles = generate_safe(custom_frag, safe_model, safe_tokenizer, safe_device, N_SAMPLES, NUM_BEAMS, MAX_LENGTH)
    if smiles:
        draw_mols(smiles)
    results.append([custom_frag] + smiles)
        
gen_smiles_df = pd.DataFrame(results, columns=['input'] + [f'top-{k}' for k in range(1, len(results[0]))])
output_dir = f'results/safe/gpt/{MODEL_VER}/{FRAG_METHOD}/beam/custom_frag/'
os.makedirs(output_dir, exist_ok=True)
gen_smiles_df.to_csv(f'{output_dir}/gen_smiles.csv', index=False)

## 7. Optional Training with the Repository Scripts

Training is not performed by any notebook cell. To train your own checkpoint,
prepare the datasets and follow the library modifications in the [README](../README.md).
Run the existing script from a terminal at the repository root:

```bash
bash src/train_model/run_safe.sh
```

Set FRAG_NAME, MODE, and PRETRAINED_DIR in that script to match this notebook's configuration.
The notebook does not change shell-script variables. Use the resulting `best_model`
directory as the model path before running inference.

The script trains on `full_safe` using `safe-train`. Fine-tuning supplies the weights
from `PRETRAINED_DIR`; from-scratch mode uses its config and tokenizer without loading
those weights. It uses the script's learning rate, epoch count, and batching settings.
`MODEL_VER="pretrained"` is an inference-only choice that bypasses local training.
